In [ ]:
#Calculate H factor
import numpy as np

def volume_integral(geqdsk,rho_old,profile):

    rho = geqdsk['RHOVN']

    profile_sized = interp(rho,rho_old,profile)

    profile_integrated = geqdsk.volume_integral(profile_sized)[-1]
    
    return profile_integrated

def taue_h_mode(x,a1=0.93,a2=0.15,a3=0.41,a4=-0.69,a5=1.97,a6=0.78,a7=0.58): #M = 2.5 for DT
    h_mode_vars = [x['ip'],x['bcentr'],x['density'],x['ptot_cps'],x['rsurf'],x['kappa'],x['eps']]
    return 0.0562*2.5**(0.19)*np.prod(np.power(h_mode_vars,np.array([a1,a2,a3,a4,a5,a6,a7])))

def taue_l_mode(x,a1=0.85,a2=1.2,a3=0.3,a4=0.5,a5=0.1,a6=0.2,a7=-0.5): #M = 2.5 for DT
    l_mode_vars = [x['ip'],x['rsurf'],x['aminor'],x['kappa'],x['density']/10,x['bcentr'],x['ptot_cps']] #this scaling law uses n20
    return 0.048*2.5**(0.5)*np.prod(np.power(x,np.array([a1,a2,a3,a4,a5,a6,a7])),axis=1)

def get_params_dict(geqdsk,gacode):
    ip = abs(gacode['IP_EXP'])
    bt = abs(gacode['BT_EXP'])
    
    rho_tor_norm = gacode['rho']
    e_density_profile = gacode['ne'] #10^19
    density_19 = volume_integral(geqdsk, rho_tor_norm, e_density_profile)
    vol = geqdsk['fluxSurfaces']['geo']['vol'][-1]
    density_19_avg = density_19/vol
    
    Palpha = (gacode['pow_e_fus'][-1] + gacode['pow_i_fus'][-1]) #multiply by 5 would give fusion power, don't do for this
    Ptot = gacode['pow_i_aux'][-1] + gacode['pow_e_aux'][-1] + Palpha
    Rmaj = geqdsk['fluxSurfaces']['geo']['R'][-1]
    kappa = geqdsk['fluxSurfaces']['geo']['kap'][-1]
    aminor = geqdsk['fluxSurfaces']['geo']['a'][-1]
    eps = geqdsk['fluxSurfaces']['geo']['eps'][-1]
    params_dict = {'ip':ip,'bcentr':bt,'density':density_19_avg,'ptot_cps':Ptot,'rsurf':Rmaj,'kappa':kappa,'eps':eps,'aminor':aminor}
    return params_dict

def calc_H98y2(gacode,geqdsk):
    total_pressure = gacode['ptot'] #do not divide by 1602.1766 (keep in Joules)
    Wthr = volume_integral(geqdsk,linspace(0,1.0,201),(3/2)*total_pressure)/1e6 #MJ
    print('Wthr =',Wthr)

    params_dict = get_params_dict(geqdsk,gacode)
    print(params_dict)
    tau_H98y2 = taue_h_mode(params_dict,a1=0.93,a2=0.15,a3=0.41,a4=-0.69,a5=1.97,a6=0.78,a7=0.58)
    print('tau_H98y2 =',tau_H98y2)
    tau_E = Wthr/params_dict['ptot_cps']
    print('tau_E =',tau_E)
    H98y2 = tau_E/tau_H98y2
    print('H98y2 =',H98y2)

    return H98y2


gacode = OMFIT['TGYRO']['OUTPUTS']['input.gacode']
geqdsk = OMFIT['CHEASE']['OUTPUTS']['EQDSK.OUT']
print(calc_H98y2(gacode,geqdsk))